# HW 2 Data Acquisition and Processing - Notebook 1

This notebook loads processed SNOTEL and USGS streamflow data for the Starvation Reservoir basin analysis. It develops the final figures used in the report, including SWE comparisons, monthly streamflow distributions, and relationships between peak SWE and monthly inflow volumes.

### Importing libraries

In [6]:
import pandas as pd
import numpy as np
from dataretrieval import nwis
import requests
import pydaymet as daymet

### Adding in SWE (Daymet - Trial Lake)

In [7]:
def get_swe():
    lat = 40.7
    lon = -110.9
    dates = ("1980-10-01", "2025-09-30")
    
    df = daymet.get_bycoords((lon, lat), dates, variables=["swe"])
    df = df.reset_index()
    
    # rename columns to match notebook style
    df = df.rename(columns={
        "time": "date",
        "swe (kg/m2)": "swe"
    })
    
    df["date"] = pd.to_datetime(df["date"])
    df["swe"] = pd.to_numeric(df["swe"], errors="coerce")
    
    return df[["date", "swe"]]

### Adding in USGS flow

In [8]:
def get_flow(site="09288180"):
    df, _ = nwis.get_dv(
        sites=site,
        parameterCd="00060",
        start="1980-10-01",
        end="2025-09-30"
    )
    
    df = df.reset_index()
    col = [c for c in df.columns if "00060_Mean" in c][0]
    
    df = df.rename(columns={"datetime": "date", col: "flow"})
    df["date"] = pd.to_datetime(df["date"])
    
    return df[["date", "flow"]]

### Add water year

In [9]:
def add_wy(df):
    df = df.copy()
    df["wy"] = np.where(df["date"].dt.month >= 10,
                        df["date"].dt.year + 1,
                        df["date"].dt.year)
    return df

### Monthly flow (acre-ft)

In [10]:
def monthly_flow(df):
    df = add_wy(df)
    df["month"] = df["date"].dt.month
    df["vol"] = df["flow"] * 1.98347
    
    return df.groupby(["wy", "month"])["vol"].sum().reset_index()

### Peak SWE

In [11]:
def peak_swe(df):
    df = add_wy(df)
    return df.groupby("wy")["swe"].max().reset_index()

### Now to run everything

In [12]:
swe = get_swe()
flow = get_flow()

swe = add_wy(swe)
flow = add_wy(flow)

monthly = monthly_flow(flow)
peak = peak_swe(swe)

# save
swe.to_csv("data/swe.csv", index=False)
flow.to_csv("data/flow.csv", index=False)
monthly.to_csv("data/monthly.csv", index=False)
peak.to_csv("data/peak_swe.csv", index=False)